<a href="https://colab.research.google.com/github/cbh4635/DL_studty/blob/main/Transformer_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# capture는 라이브러리 설치 시 출력이 나오지 않게함
%%capture

# 그래프에서 한글이 깨지지 않게 한글 폰트 설치
!sudo apt-get install -y fonts-nanum # 나눔 폰트
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 적용
import matplotlib.pyplot as plt
plt.rc('font', family='NanumBarunGothic')

# 맨처음에 셀 한 번 실행하고 세션 다시 시작하고 또 실행해야 반영됨
# 런타임 -> 세션 다시시작

In [2]:
%%capture

# 트랜스포머 구현을 위한 라이브러리 설치 (Hugging Face)
!pip install transformers # https://wikidocs.net/166691
!pip install sentencepiece # MarianTokenizer 불러올 때 필요
!pip install sacremoses # MarianMTModel 에서 불러올 때 warning 뜨는 것 방지
!pip install einops # Einstein operations => rearrange 함수를 위해

In [3]:
import torch
from torch import nn, optim

from transformers import MarianMTModel, MarianTokenizer # MT: Machine Translation
from tqdm import tqdm
import math, random
from einops import rearrange

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [6]:
# for random seed
random_seed = 0 # 랜덤 결과가 아닌 고정으로 설정 (실습시 동일 결과 출력을 위해)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed) # if use multi-GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
random.seed(random_seed)

In [ ]:
# Load the tokenizer & model
tokenizer = MarianTokenizer.from_pretrained('Helsinki-NLP/opus-mt-ko-en') #한국어-영어 번역을 위한 토크나이저
model = MarianMTModel.from_pretrained('Helsinki-NLP/opus-mt-ko-en') # MT: Machine Translation

## 토크나이저 사용해보기

In [ ]:
eos_idx = tokenizer.eos_token_id
pad_idx = tokenizer.pad_token_id
print("eos_idx = ", eos_idx)
print("pad_idx = ", pad_idx)

In [ ]:
# 토크나이저 구현 방식에 대해 참고 자료: https://ratsgo.github.io/nlpbook/docs/preprocess/bpe/
# '_'로 띄어쓰기를 나타낸다! 즉, '_'가 없으면 이어진 한 단어임을 나타냄 (subword tokenizing)
print(tokenizer.tokenize("Hi, I'm Choi. ...  Hello?"))
print(tokenizer.tokenize("a/b 1+2+3 2:1 a>b"))
print(tokenizer.tokenize("pretrained restart"))
print(tokenizer.tokenize("chatGPT"))
print(tokenizer.tokenize("The example is very good in our lecture")) # 띄어쓰기 자체도 tokenize 함
print(tokenizer.tokenize("한글은 어떻게 할까?"))
print(tokenizer.tokenize("띄어쓰기 기준으로 토크나이징을 하진 않는 것 같다."))

In [ ]:
print(tokenizer.get_vocab())
vocab_size = tokenizer.vocab_size
print(vocab_size)

In [ ]:
# add_special_tokens=False 는 <eos> 자동 붙여주는 것을 방지 - True(default)면 마지막에 <eos> 를 붙임

# 'encode' 매서드
print(tokenizer.encode('<pad>', add_special_tokens=False)) # <pad>는 65000
print(tokenizer.encode('</s>', add_special_tokens=False)) # <eos>는 0 (해당 토크나이저는 <sos> 가 따로 정의 안됨)

 # 대소문자 다른 단어로 인식
print(tokenizer.encode('He', add_special_tokens=False))
print(tokenizer.encode('he', add_special_tokens=False))

print(tokenizer.encode('인공', add_special_tokens=False))
print(tokenizer.encode('지능', add_special_tokens=False))
print(tokenizer.encode('인공지능', add_special_tokens=False))

In [ ]:
# 'tokenize'와 'encode'의 차이
print(tokenizer.tokenize('문장을 넣으면 토크나이즈해서 숫자로 바꾼다'))
print(tokenizer.encode('문장을 넣으면 토크나이즈해서 숫자로 바꾼다', add_special_tokens=False))

In [ ]:
# decode 매서드
print(tokenizer.decode([204]))
print(tokenizer.decode([206]))
print(tokenizer.decode([210]))

In [ ]:
# 사전 학습된 모델(MarianMTModel)로 번역해보기
input_text = "자율주행 차량의 핵심 기술은 무엇인가?"

# step.1 입력 텍스트 encoding
input_tokens = tokenizer.encode(input_text, return_tensors="pt") # pt: pytorch, np: numpy

# step.2 model 예측 수행
translated_tokens = model.generate(input_tokens, max_new_tokens=100) # max_new_tokens: 최대 토큰 수

# step.2 예측 결과 decoding
translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

print("입력:", input_text)
print("AI의 번역:", translated_text)

## DS, DL 생성 & 테스트

In [ ]:
# git clone하여 dataset 및 pre-train 파라미터 불러오기
!git clone https://github.com/cbh4635/TF_SFLG

In [16]:
# Pandas 라이브러리를 이용하여 Excel파일로 부터 DataFrame을 생성
import pandas as pd

# Dataset (한국어-영어 번역 데이터셋)
# https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&aihubDataSe=realm&dataSetSn=126 에서 다운 가능 (AI-Hub)
data = pd.read_excel('/content/TF_SFLG/kor_chat_data.xlsx')

In [ ]:
# DataFrame 살펴보기
data

In [18]:
data_idx = 100
print(f'input: {data.loc[data_idx,'원문']}')
print(f'output: {data.loc[data_idx,'번역문']}')

input: 지난주 금요일에 열린 자선행사의 반응은 어땠나요?
output: How did the charity event go last Friday?


In [19]:
# 데이터셋 클래스 정의
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data.loc[idx, '원문'], self.data.loc[idx, '번역문']

custom_DS = CustomDataset(data)

In [ ]:
print(custom_DS.data.shape)

print(len(custom_DS))

src, trg = custom_DS[0]
print(src)
print(trg)

In [ ]:
# Transformer 논문에서는 450만개 영,독 문장 pair 사용
train_DS, val_DS, test_DS = torch.utils.data.random_split(custom_DS, [97000, 2000, 1000])

print('==== DataSet ====')
print(f'train_DS: {len(train_DS)}')
print(f'val_DS:   {len(val_DS)}')
print(f'test_DS:  {len(test_DS)}')

In [ ]:
BATCH_SIZE = 64
max_len = 100

# DataLoader 생성
train_DL = torch.utils.data.DataLoader(train_DS, batch_size=BATCH_SIZE, shuffle=True)
val_DL = torch.utils.data.DataLoader(val_DS, batch_size=BATCH_SIZE, shuffle=True)
test_DL = torch.utils.data.DataLoader(test_DS, batch_size=BATCH_SIZE, shuffle=True)

print('\n==== DataLoader ====')
print(f'train_DL: {len(train_DL)}')
print(f'val_DL:   {len(val_DL)}')
print(f'test_DL:  {len(test_DL)}')

# DataLoader 확인
src_texts, trg_texts = next(iter(test_DL))
print('\nData check:')
print(src_texts)
print(trg_texts)
print(len(src_texts))
print(len(trg_texts))

# 여러 문장에 대해서는 tokenizer.encode() 가 아닌 그냥 tokenizer()
# truncation = True: max_len 보다 길면 끊고 <eos> 집어넣어버림
src = tokenizer(src_texts, padding=True, truncation=True, max_length = max_len, return_tensors='pt', add_special_tokens = False).input_ids

# <sos>가 토크나이저에 따로 없어서 </s> 를 <sos> 로 사용
trg_texts = ['</s> ' + s for s in trg_texts]
trg = tokenizer(trg_texts, padding=True, truncation=True, max_length = max_len, return_tensors='pt', add_special_tokens = True).input_ids

print('\n원문')
print(src[:2])
print(src.shape)

print('\n번역')
print(trg[:2])
print(trg.shape)

In [ ]:
# 번역 문장의 마지막 token 확인
print(trg[:,-1]) # 가장 마지막 단어를 보니 어떤 문장은 <eos> 로 끝이 났고 나머지는 <pad> 로 끝이 났다는 걸 볼 수 있음
# 마지막 token이 eos인 가장 긴 문장을 찾음
longest_trg = (trg[:,-1] == eos_idx)
print(longest_trg, '\n')

longest_idx = longest_trg.nonzero()[0][0] # 가장 긴 문장 중 첫 번째 문장 관찰 (한개가 아닐 수 도 있음)
print(f'Longest sentence: {len(trg[longest_idx,:])}')
print(tokenizer.decode(trg[longest_idx,:]),'\n')

# 디코더 입력/출력(GT)
print('Longest in/out:')
print(trg[longest_idx,:-1])
print(trg[longest_idx,1:],'\n')

short_idx = (~longest_trg).nonzero()[0][0]
print('Short in/out:')
print(trg[short_idx,:-1])
print(trg[short_idx,1:])

## Transformer 직접 구현하기

### 하이퍼 파라미터 정의

In [33]:
# 논문에 나오는 base 모델 (train loss를 많이 줄이려면 많은 Epoch이 요구됨, 또, test 성능도 좋으려면 더 많은 데이터 요구)
# n_layers = 6
# d_model = 512
# d_ff = 2048
# n_heads = 8
# drop_p = 0.1

# 사이즈를 줄인 경량화 모델
n_layers = 3
d_model = 256
d_ff = 512  # FFN dim
n_heads = 8
drop_p = 0.1 # Dropout 비율

### Multi-Head Attention 구현하기

In [72]:
class MHA(nn.Module): # Multi Head Attention
    def __init__(self, d_model=256, n_heads=8):
        """
        d_model: 모델의 차원
        n_heads: 헤드 수
        """
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model / n_heads

        # Embedding Vector Projection
        self.fc_q = nn.Linear(d_model, d_model)
        self.fc_k = nn.Linear(d_model, d_model)
        self.fc_v = nn.Linear(d_model, d_model)

        # Output Projection
        self.fc_o = nn.Linear(d_model, d_model)

        self.scale = torch.sqrt(torch.tensor(self.head_dim)) # sqrt(dim_k)

    def forward(self, Q, K, V, mask = None):
        """
        (B: Batch size, N: Max lengh of tokens, C: Channel dimension, H: Number of heads)
        Q : Query, shape = [B, N_q, C]
        K : Key, shape = [B, N_k, C]
        V : Value, shape = [B, N_k, C]
        mask : Attention padding mask, shape = [B, H, N_q, N_k]
        """
        Q = self.fc_q(Q) # [B, N, C]
        K = self.fc_k(K) # [B, N, C]
        V = self.fc_v(V) # [B, N, C]

        # Multi-head 분할 및 QK^T 계산을 위한 shape 변경

        # einops 라이브러리의 'rearrange' 활용
        Q = rearrange(Q, 'B N (H c) -> B H N c', H = self.n_heads) # c = self.head_dim
        K = rearrange(K, 'B N (H c) -> B H N c', H = self.n_heads)
        V = rearrange(V, 'B N (H c) -> B H N c', H = self.n_heads)

        # 참고: 일반적인 구현방법 ('rearrange' 활용 X)
        # batch_size = Q.shape[0]
        # Q = Q.reshape(batch_size, -1, self.n_heads, self.head_dim).permute(0,2,1,3)

        # Attention score 계산 - softmax(QK^T)
        attention_score = Q @ K.transpose(-2,-1) / self.scale # [B, H, N_q, N_k] = [B, H, N_q, c] @ [B, H, c, N_k]
        # 아래 방식으로도 구현 가능
        # attention_score = torch.matmul(Q, K.transpose(-2, -1)) / self.scale

        if mask is not None:
            attention_score[mask] = float('-inf')

        attention_weights = torch.softmax(attention_score, dim=-1) # [B, H, N_q, N_k]

        attention_out = attention_weights @ V # [B, H, N_q, c] = [B, H, N_q, N_k] @ [B, H, N_k, c]

        attention_out = rearrange(attention_out, 'B H N c -> B N (H c)') # [B, N, C]
        attention_out = self.fc_o(attention_out)  # [B, N, C]

        return attention_out, attention_weights


class FeedForward(nn.Module): # Feed Forward Network(FFN)
    def __init__(self, d_model, d_ff, drop_p):
        super().__init__()

        self.linear = nn.Sequential(nn.Linear(d_model, d_ff),
                                    nn.ReLU(),
                                    nn.Dropout(drop_p),
                                    nn.Linear(d_ff, d_model))

    def forward(self, x):
        x = self.linear(x)
        return x

In [ ]:
# Scaling을 위해 sqrt(dim_k)로 나누는 이유 살펴보기
C = torch.arange(4, 256, 2) # 차원 수
N = 1000 # 토큰 수
result_og=[]
result_sc=[]
for c in C:
    # 내적값(attention score) 계산
    inner_prod = torch.randn(N,c) @ torch.randn(N,c).T

    # 내적값을 구해서 variance를 구함
    result_og += [torch.var(inner_prod)]
    result_sc += [torch.var(inner_prod/torch.sqrt(c))]

plt.plot(C,result_og, 'b')
plt.plot(C,result_sc, 'r')

In [ ]:
# scaling을 위해 sqrt(dim_k)로 나누는 이유: 안나누면 softmax로 들어가는 입력의 분산이 커짐 (차원이 커질수록 분산이 선형적으로 증가)
# 입력의 분산이 커지면 softmax의 결과가 극단적으로 나뉘어짐 -> gradient도 작아져 학습에 악영향
# 즉, feature의 차원(dim_k)이 커지면 variance도 같이 커지는 것을 방지하기 위해 std(=sqrt(dim_k))로 나눠서 dim_k에 의한 영향성을 제거
softmax_in = torch.arange(-10,11).float()
print(f'Scaling O: \n{torch.softmax(softmax_in, dim=-1)}\n')
print(f'Scaling X: \n{torch.softmax(softmax_in/math.sqrt(len(softmax_in)), dim=-1)}')

### Encoder 구현

In [60]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, drop_p):
        super().__init__()

        self.self_atten = MHA(d_model, n_heads)
        self.self_atten_LN = nn.LayerNorm(d_model)

        self.FF = FeedForward(d_model, d_ff, drop_p)
        self.FF_LN = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(drop_p)

    def forward(self, x, enc_mask):

        residual, atten_enc = self.self_atten(x, x, x, enc_mask)
        residual = self.dropout(residual)
        x = self.self_atten_LN(x + residual)

        residual = self.FF(x)
        residual = self.dropout(residual)
        x = self.FF_LN(x + residual)
        return x, atten_enc


class Encoder(nn.Module):
    def __init__(self, input_embedding, max_len, n_layers, d_model, d_ff, n_heads, drop_p):
        """
        input_embedding: 입력 token에 대한 임베딩 레이어
        """
        super().__init__()

        self.scale = math.sqrt(d_model)
        self.input_embedding = input_embedding

        # Learnable embedding으로 positional encoding을 구현
        # N개의 입력에 대해 C개의 파라미터를 가짐 (총 파라미터수 N*C개)
        self.pos_embedding = nn.Embedding(max_len, d_model) # [B, N] -> [B, N, C] (Batch간 파라미터는 공유됨)
        # 논문에서는 Sine/Cosine기반 encoding 사용 - 아래 코드 참고
        # https://github.com/tatp22/multidim-positional-encoding/blob/master/positional_encodings/torch_encodings.py

        self.dropout = nn.Dropout(drop_p)

        self.layers = nn.ModuleList([EncoderLayer(d_model, d_ff, n_heads, drop_p) for _ in range(n_layers)])

    def forward(self, src, mask, atten_map_save = False):
        """
        src: Input tokens, shape = [B, N]
        mask: Attention padding masking, shape = [B, H, N, N]
        """
        pos = torch.arange(src.shape[1]).expand_as(src).to(DEVICE) # [N] -> [B, N]

        # self.scale 을 곱하여 position 보다 token 정보(feature)에 비중을 높힘
        x = self.scale * self.input_embedding(src) + self.pos_embedding(pos) # [B, N, C]

        x = self.dropout(x)

        atten_encs = torch.tensor([]).to(DEVICE)
        for layer in self.layers:
            x, atten_enc = layer(x, mask)
            if atten_map_save is True:
                atten_encs = torch.cat([atten_encs , atten_enc[0].unsqueeze(0)], dim=0) # 첫번째 문장의 attention 결과 저장

        return x, atten_encs

### Decoder 구현

In [68]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_heads, drop_p):
        super().__init__()

        self.self_atten = MHA(d_model, n_heads)
        self.self_atten_LN = nn.LayerNorm(d_model)

        self.enc_dec_atten = MHA(d_model, n_heads) # Cross-attention
        self.enc_dec_atten_LN = nn.LayerNorm(d_model)

        self.FF = FeedForward(d_model, d_ff, drop_p)
        self.FF_LN = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(drop_p)

    def forward(self, x, enc_out, dec_mask, enc_dec_mask):

        residual, atten_dec = self.self_atten(x, x, x, dec_mask)
        residual = self.dropout(residual)
        x = self.self_atten_LN(x + residual)

        # Query는 디코더로부터 Key,Value는 인코더로부터
        residual, atten_enc_dec = self.enc_dec_atten(x, enc_out, enc_out, enc_dec_mask)
        residual = self.dropout(residual)
        x = self.enc_dec_atten_LN(x + residual)

        residual = self.FF(x)
        residual = self.dropout(residual)
        x = self.FF_LN(x + residual) # [B, N, C]

        return x, atten_dec, atten_enc_dec

class Decoder(nn.Module):
    def __init__(self, input_embedding, max_len, n_layers, d_model, d_ff, n_heads, drop_p):
        super().__init__()

        # self.scale = torch.sqrt(torch.tensor(d_model))
        self.scale = math.sqrt(d_model)
        self.input_embedding = input_embedding
        self.pos_embedding = nn.Embedding(max_len, d_model)

        self.dropout = nn.Dropout(drop_p)

        self.layers = nn.ModuleList([DecoderLayer(d_model, d_ff, n_heads, drop_p) for _ in range(n_layers)])

        # 각 token(총 vocab_size개)에 대한 확률을 예측함
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, trg, enc_out, dec_mask, enc_dec_mask, atten_map_save = False):
        """
        trg: target tokens, shape = [B, N_dec]
        enc_out: encoder output, shape = [B, N_enc, C]
        dec_mask: decoder self-attention padding shape = [B, H, N_dec, N_dec]
        enc_dec_mask: encoder-decoder(cross) attention padding masking, shape = [B, H, N_dec, N_enc]
        """
        pos = torch.arange(trg.shape[1]).expand_as(trg).to(DEVICE)

        x = self.scale*self.input_embedding(trg) + self.pos_embedding(pos)
        x = self.dropout(x)

        atten_decs = torch.tensor([]).to(DEVICE)
        atten_enc_decs = torch.tensor([]).to(DEVICE)
        for layer in self.layers:
            x, atten_dec, atten_enc_dec = layer(x, enc_out, dec_mask, enc_dec_mask)
            if atten_map_save is True:
                atten_decs = torch.cat([atten_decs , atten_dec[0].unsqueeze(0)], dim=0)
                atten_enc_decs = torch.cat([atten_enc_decs , atten_enc_dec[0].unsqueeze(0)], dim=0)

        x = self.fc_out(x)

        return x, atten_decs, atten_enc_decs

### 최종 Transformer 모델 구현

In [66]:
class Transformer(nn.Module):
    def __init__(self, vocab_size, max_len, n_layers, d_model, d_ff, n_heads, drop_p):
        super().__init__()

        self.input_embedding = nn.Embedding(vocab_size, d_model) # 각 token에 대해 다른 임베딩 값을 가지도록 학습됨
        self.encoder = Encoder(self.input_embedding, max_len, n_layers, d_model, d_ff, n_heads, drop_p)
        self.decoder = Decoder(self.input_embedding, max_len, n_layers, d_model, d_ff, n_heads, drop_p)

        self.n_heads = n_heads

        for m in self.modules():
            if hasattr(m,'weight'):
                # LayerNorm(dim=1)에 대해선 추가 initial 안하겠다는 뜻 (LayerNorm은 “identity로 시작”하도록 설계됨)
                if m.weight.dim() > 1:
                    # xavier 초기화: 각 layer의 입출력 파라미터 수에 맞춰 분산을 adaptive하게 결정 하도록함
                    nn.init.xavier_uniform_(m.weight) # xavier의 분산은 2/(Nin+Nout) (Transformer 기반 모델 초기화에 활용)
                    # 참고: kaiming 초기화 (입력 파라미터 수에 집중)
                    # nn.init.kaiming_uniform_(m.weight) # kaiming의 분산은 2/Nin (CNN기반 모델 초기화에 활용) - xavier에 비해 분산이 큼

    def make_enc_mask(self, src):

        enc_mask = (src == pad_idx).unsqueeze(1).unsqueeze(2) # [B, 1, 1, N_enc]
        enc_mask = enc_mask.expand(src.shape[0], self.n_heads, src.shape[1], src.shape[1]) # [B, H, N_enc, N_enc]
        """ src pad mask (문장 마다 다르게 생김. 이건 한 문장에 대한 pad 행렬)
        F F T T
        F F T T
        F F T T
        F F T T
        """
        return enc_mask

    def make_dec_mask(self, trg):

        dec_mask = ~(torch.tril(torch.ones(trg.shape[0], self.n_heads, trg.shape[1], trg.shape[1], dtype=torch.bool))) # [B, H, N_dec, N_dec]
        """ trg future mask
        F T T T T
        F F T T T
        F F F T T
        F F F F T
        F F F F F
        """
        return dec_mask

    def make_enc_dec_mask(self, src, trg):

        enc_dec_mask = (src == pad_idx).unsqueeze(1).unsqueeze(2) # [B, 1, 1, N_enc]
        enc_dec_mask = enc_dec_mask.expand(trg.shape[0], self.n_heads, trg.shape[1], src.shape[1]) # [B, H, N_dec, N_enc]
        """ src pad mask
        F F T T
        F F T T
        F F T T
        F F T T
        F F T T
        """
        return enc_dec_mask

    def forward(self, src, trg):

        enc_mask = self.make_enc_mask(src)
        dec_mask = self.make_dec_mask(trg)
        enc_dec_mask = self.make_enc_dec_mask(src, trg)

        enc_out, atten_encs = self.encoder(src, enc_mask)
        out, atten_decs, atten_enc_decs = self.decoder(trg, enc_out, dec_mask, enc_dec_mask)

        return out, atten_encs, atten_decs, atten_enc_decs

### 모델 생성

In [69]:
model = Transformer(vocab_size, max_len, n_layers, d_model, d_ff, n_heads, drop_p).to(DEVICE)

# 출력 결과 체크
b = 2
len_enc = 8
len_dec = 10

src = torch.randint(vocab_size, (b,len_enc)).to(DEVICE)
trg = torch.randint(vocab_size, (b,len_dec)).to(DEVICE)

model.eval()
with torch.no_grad():
    x = model(src, trg)[0]

print(trg.shape)
print(x.shape)

torch.Size([2, 10])
torch.Size([2, 10, 65001])


### Train, Test, loss_epoch 함수 정의

In [43]:
def Train(model, train_DL, val_DL, criterion, optimizer, scheduler = None):
    loss_history = {"train": [], "val": []}
    best_loss = 9999
    for ep in range(EPOCH):
        model.train() # train mode로 전환
        train_loss = loss_epoch(model, train_DL, criterion, optimizer = optimizer, scheduler = scheduler)
        loss_history["train"] += [train_loss]

        model.eval() # test mode로 전환
        with torch.no_grad():
            val_loss = loss_epoch(model, val_DL, criterion)
            loss_history["val"] += [val_loss]

            # validation loss가 가장 작을 때 저장
            if val_loss < best_loss:
                best_loss = val_loss
                torch.save({"model": model,
                            "ep": ep,
                            "optimizer": optimizer,
                            "scheduler": scheduler,}, save_model_path)
        # print loss
        print(f"Epoch {ep+1}: train loss: {train_loss:.5f}   val loss: {val_loss:.5f}   current_LR: {optimizer.param_groups[0]['lr']:.8f}")
        print("-" * 20)

    torch.save({"loss_history": loss_history,
                "EPOCH": EPOCH,
                "BATCH_SIZE": BATCH_SIZE}, save_history_path)

def Test(model, test_DL, criterion):
    model.eval() # test mode로 전환
    with torch.no_grad():
        test_loss = loss_epoch(model, test_DL, criterion)
    print(f"Test loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):.3f}")

# PPL(Perplexity) -“이 모델이 다음 토큰을 얼마나 헷갈려 하는지”를 나타내는 지표 → 낮을수록 좋음
# 예시: PPL = 1 → 완벽하게 예측,  PPL = 10 → 매 토큰마다 평균적으로 10개 중 하나를 고민하는 수준

def loss_epoch(model, DL, criterion, optimizer = None, scheduler = None):
    N = len(DL.dataset) # the number of data

    rloss=0
    for src_texts, trg_texts in tqdm(DL, leave=False): # 'tqdm'라이브러리: Progress을 시각화
        src = tokenizer(src_texts, padding=True, truncation=True, max_length = max_len, return_tensors='pt', add_special_tokens = False).input_ids.to(DEVICE)
        trg_texts = ['</s> ' + s for s in trg_texts]
        trg = tokenizer(trg_texts, padding=True, truncation=True, max_length = max_len, return_tensors='pt').input_ids.to(DEVICE)

        # 모델 inference
        y_hat = model(src, trg[:,:-1])[0] # 모델 통과 시킬 땐 trg의 마지막 토큰(eos)은 제외

        # loss 계산
        # loss 계산 시엔 <sos> 는 제외, [B, N, C] -> [B, C, N]으로 바꿔줌 (loss 함수가 그렇게 구현되어 있음)
        loss = criterion(y_hat.permute(0,2,1), trg[:,1:])

        # update
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        # loss accumulation
        loss_b = loss.item() * src.shape[0]
        rloss += loss_b
    loss_e = rloss/N
    return loss_e

def count_params(model):
    num = sum([p.numel() for p in model.parameters() if p.requires_grad])
    return num

class NoamScheduler:
    def __init__(self, optimizer, d_model, warmup_steps, LR_scale = 1):
        self.optimizer = optimizer
        self.current_step = 0
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.LR_scale = LR_scale

    def step(self):
        self.current_step += 1
        lrate = self.LR_scale * (self.d_model ** -0.5) * min(self.current_step ** -0.5, self.current_step * self.warmup_steps ** -1.5)
        self.optimizer.param_groups[0]['lr'] = lrate


## 모델 학습

In [45]:
new_model_train = False

if new_model_train: # 새로 학습할 모델 파라미터 경로
    save_model_path = 'Transformer_small_new.pt'
    save_history_path = 'Transformer_small_history_new.pt'

else: # 미리 학습해둔 모델 파라미터 경로
    save_model_path = '/content/TF_SFLG/Transformer_small.pt'
    save_history_path = '/content/TF_SFLG/Transformer_small_history.pt'

# 학습 파라미터 설정
EPOCH = 15
criterion = nn.CrossEntropyLoss(ignore_index = pad_idx) # pad token 이 출력 나와야하는 시점의 loss는 무시 (즉, label이 <pad> 일 때는 무시)
warmup_steps = 1000
LR_scale = 0.5

In [46]:
if new_model_train:
    params = [p for p in model.parameters() if p.requires_grad] # 사전 학습된 layer를 사용할 경우
    optimizer = optim.Adam(params, lr=0, # 맨 처음 step 의 LR=0으로 출발 (warm-up)
                            betas=(0.9, 0.98), eps=1e-9) # 논문에서 제시한 beta와 eps 사용
    scheduler = NoamScheduler(optimizer, d_model=d_model, warmup_steps=warmup_steps, LR_scale=LR_scale)

    Train(model, train_DL, val_DL, criterion, optimizer, scheduler)

## 모델  불러오기

In [ ]:
loaded = torch.load(save_model_path, map_location=DEVICE, weights_only=False)
load_model = loaded["model"]
ep = loaded["ep"]
optimizer = loaded["optimizer"]

loaded = torch.load(save_history_path, map_location=DEVICE)
loss_history = loaded["loss_history"]

print(ep)
print(optimizer)
print(count_params(load_model))

In [ ]:
plt.figure()
plt.plot(range(1,EPOCH+1),loss_history["train"], label="train")
plt.plot(range(1,EPOCH+1),loss_history["val"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train, Val Loss")
plt.grid()
plt.legend()

In [ ]:
# 모델 성능 테스트
Test(load_model, test_DL, criterion)

## Translation 함수, Attention map Viz 함수 정의

In [82]:
def translation(model, src_text, atten_map_save = False):
    model.eval()
    with torch.no_grad():
        src = tokenizer.encode(src_text, return_tensors='pt', add_special_tokens=False).to(DEVICE)
        enc_mask = model.make_enc_mask(src) # 문장 하나만 넣을거라 사실상 안해도 됨
        enc_out, atten_encs = model.encoder(src, enc_mask, atten_map_save)

        pred = tokenizer.encode('</s>', return_tensors='pt', add_special_tokens=False).to(DEVICE) # 1x1
        for _ in range(max_len-1): # <sos> 가 한 토큰이기 때문에 최대 99 번까지만 loop을 돌아야 함
            dec_mask = model.make_dec_mask(pred) # 추론에서는 사실상 안해도 됨
            enc_dec_mask = model.make_enc_dec_mask(src, pred)
            out, atten_decs, atten_enc_decs = model.decoder(pred, enc_out, dec_mask, enc_dec_mask, atten_map_save)
            # out.shape = (B=1, N, vocab_size)

            pred_word = out[:,-1,:].argmax(dim=1).unsqueeze(0) # 마지막 단어(토큰)에 대해 argmax해서 prediction함

            if tokenizer.decode(pred_word.item()) == '</s>':
                break

            pred = torch.cat([pred, pred_word], dim=1) # 단어 하나씩 추가

        translated_text = tokenizer.decode(pred[0,1:])

    return translated_text, atten_encs, atten_decs, atten_enc_decs


def show_attention(atten, Query, Key, n):
    atten = atten.cpu()

    # Query/Key 길이
    q_len = len(Query)
    k_len = len(Key)

    # attention의 실제 크기
    # atten shape 예: (B, H, Q, K)
    Q = atten.shape[2]
    K = atten.shape[3]

    # 둘 중 작은 길이로 맞춤
    q = min(q_len, Q)
    k = min(k_len, K)

    # attention 잘라내기
    atten = atten[:, :, :q, :k]

    fig, ax = plt.subplots(1, 3, figsize=[k * 1.5, q])
    for i in range(3): # n 번째 layer, 앞 세 개의 헤드만 plot
        ax[i].set_yticks(range(q))
        ax[i].set_yticklabels(Query[:q], rotation=45)
        ax[i].set_xticks(range(k))
        ax[i].set_xticklabels(Key[:k], rotation=60)
        ax[i].imshow(atten[n-1][i], cmap="bone")

In [ ]:
# 번역해보기
src_text, trg_text = test_DS[2]
print(f"입력: {src_text}")
print(f"정답: {trg_text}")

translated_text, atten_encs, atten_decs, atten_enc_decs = translation(load_model, src_text, atten_map_save = True)
print(f"AI의 번역: {translated_text}")

In [ ]:
enc_input = tokenizer.tokenize(src_text)
dec_tokens = tokenizer.tokenize(translated_text)
dec_input = dec_tokens[:-1] # 디코더 입력으로 들어가는 문장(sos 는 있고 eos는 없고)
dec_output = dec_tokens[1:] # 디코더 출력으로 나간 문장(eos 는 있고 sos는 없고)

show_attention(atten_encs, enc_input, enc_input, n = 1)
show_attention(atten_decs, dec_input, dec_input, n = 3)
show_attention(atten_enc_decs, dec_output, enc_input, n = 3)

In [ ]:
# 직접 구현한 번역기 사용
src_text = "아 퇴근하고 싶다."
print(f"입력: {src_text}")

translated_text = translation(load_model, src_text)[0]
print(f"AI의 번역: {translated_text}")